# 🌟 Sindhi Digital Classroom (SDC) — AI Model Fine-Tuning Pipeline
### Train a Domain-Specific Sindhi Science Teacher Model using QLoRA & Qwen2.5
**Curriculum:** Sindh Textbook Board (STBB) Class 6 General Science (8 Units, 90 Chunks, 201 Glossary Terms, 39 Exercises)
**Hardware Target:** Raspberry Pi 5 (8 GB RAM, ARM64 CPU) via 4-bit GGUF quantization
**Colab Requirement:** Free Tier T4 GPU (Runtime -> Change runtime type -> T4 GPU)

## Step 1: Install Dependencies
Installs Hugging Face PEFT, TRL, BitsAndBytes, and Accelerate.

In [ ]:
!pip install -q torch transformers datasets peft bitsandbytes trl accelerate sentencepiece

## Step 2: Upload Dataset
Upload `sdc_science_alpaca_dataset.json` (generated from `src/data/training/` in the SDC repo).

In [ ]:
from google.colab import files
import os

if not os.path.exists('sdc_science_alpaca_dataset.json'):
    print('Please select and upload sdc_science_alpaca_dataset.json:')
    uploaded = files.upload()
else:
    print('Dataset already found!')

## Step 3: Run Model Fine-Tuning (QLoRA 4-bit)
Fine-tunes `Qwen/Qwen2.5-0.5B-Instruct` on the STBB Sindhi science dataset.

In [ ]:
import json
import torch
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig

MODEL_ID = 'Qwen/Qwen2.5-0.5B-Instruct'
OUTPUT_DIR = 'sdc-qwen2.5-0.5b-sindhi-science'

with open('sdc_science_alpaca_dataset.json', 'r', encoding='utf-8') as f:
    raw_data = json.load(f)
print(f'Loaded {len(raw_data)} training pairs!')

SYSTEM_PROMPT = 'توهان سنڌ ٽيڪسٽ بڪ بورڊ (STBB) جا هڪ ماهر ۽ مهربان سنڌي استاد آهيو، جيڪي ڪلاس 6 جي شاگردن کي سائنس پڙهائيندا آهن۔'

formatted = []
for d in raw_data:
    u = d['instruction']
    if d.get('input'):
        u = f"حوالو يا مواد:\n{d['input']}\n\nسوال:\n{u}"
    formatted.append({'messages': [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user', 'content': u},
        {'role': 'assistant', 'content': d['output']}
    ]})

hf_dataset = Dataset.from_list(formatted)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    torch_dtype=torch.float16,
    device_map='auto',
    trust_remote_code=True
)
model = prepare_model_for_kbit_training(model)

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
    bias='none',
    task_type='CAUSAL_LM'
)
model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=4,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    fp16=True,
    bf16=False,
    logging_steps=10,
    save_strategy='epoch',
    max_length=512
)

trainer = SFTTrainer(
    model=model,
    train_dataset=hf_dataset,
    args=training_args,
    formatting_func=lambda ex: tokenizer.apply_chat_template(ex['messages'], tokenize=False)
)

print('Training starting on Colab GPU...')
trainer.train()
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print('Training complete! Saved adapter to:', OUTPUT_DIR)

## Step 4: Test Neural Generation in Sindhi
Prompt the newly fine-tuned model with a science question.

In [ ]:
test_prompt = 'گهرڙو (Cell) ڇا آهي ۽ جاندارن لاءِ ڇو ضروري آهي؟'
prompt_chat = [
    {'role': 'system', 'content': SYSTEM_PROMPT},
    {'role': 'user', 'content': test_prompt}
]
text = tokenizer.apply_chat_template(prompt_chat, tokenize=False, add_generation_prompt=True)
inputs = tokenizer([text], return_tensors='pt').to('cuda')

outputs = model.generate(
    **inputs,
    max_new_tokens=200,
    temperature=0.3,
    repetition_penalty=1.1,
    pad_token_id=tokenizer.eos_token_id
)
gen_text = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
print('Generated Sindhi Response:\n')
print(gen_text)

## Step 5: Merge Adapter & Export 4-bit GGUF for Raspberry Pi 5
Quantizes the model to GGUF format for instant execution on Raspberry Pi 5 CPU without GPU.

In [ ]:
from peft import PeftModel
import subprocess

MERGED_DIR = 'sdc-qwen2.5-0.5b-sindhi-science-merged'
GGUF_FILE = 'sdc-qwen2.5-0.5b-sindhi-science-q4_k_m.gguf'

print('Merging LoRA weights into FP16 base model...')
base = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.float16, device_map='auto')
merged = PeftModel.from_pretrained(base, OUTPUT_DIR).merge_and_unload()
merged.save_pretrained(MERGED_DIR)
tokenizer.save_pretrained(MERGED_DIR)

print('Cloning llama.cpp to build GGUF...')
!git clone https://github.com/ggerganov/llama.cpp.git
!pip install -r llama.cpp/requirements.txt
!python llama.cpp/convert_hf_to_gguf.py {MERGED_DIR} --outfile {MERGED_DIR}.f16.gguf --outtype f16
!make -C llama.cpp -j4 llama-quantize
!llama.cpp/llama-quantize {MERGED_DIR}.f16.gguf {GGUF_FILE} Q4_K_M

print('GGUF quantization complete! Downloading model file...')
from google.colab import files
files.download(GGUF_FILE)